# Vehicle Type Classification — Random Forest

Runs end to end and writes `submissions/submission_random_forest.csv`.

Many decorrelated trees averaged: much lower variance than a single tree. Randomised search over the main hyper-parameters.

## Setup and preprocessing
* Drop `vehicle_id` (identifier only).
* Drop the 2,500 exact duplicate training rows: they would leak across CV folds and inflate scores (labels never conflict, so nothing is lost).
* No imputation (no missing values), no outlier removal (bounded values), no resampling (balanced classes).
* No scaling needed (tree-based).

In [1]:
import os, json, numpy as np, pandas as pd, warnings
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings("ignore")
os.makedirs("submissions", exist_ok=True)
SEED = 42; np.random.seed(SEED)
for d in ["data", ".", "..", "/kaggle/input/vehicle-type-classification"]:
    if os.path.exists(f"{d}/vehicle_train_10000.csv"): DATA_DIR = d; break
else: raise FileNotFoundError("Put the competition CSVs next to this notebook or in ./data")
train  = pd.read_csv(f"{DATA_DIR}/vehicle_train_10000.csv")
test   = pd.read_csv(f"{DATA_DIR}/vehicle_test.csv")
sample = pd.read_csv(f"{DATA_DIR}/vehicle_sample_submission.csv")
TARGET, ID = "vehicle_class", "vehicle_id"
FEATURES = [c for c in test.columns if c != ID]
df = train.drop_duplicates(subset=FEATURES).reset_index(drop=True)   # remove 2,500 exact duplicate rows
X, y, X_test = df[FEATURES], df[TARGET].values, test[FEATURES]
CV5 = StratifiedKFold(5, shuffle=True, random_state=SEED)
print("training rows after de-dup:", len(df), "| class counts:", np.bincount(y))

training rows after de-dup: 7500 | class counts: [2539 2490 2471]


## Hyper-parameter tuning (5-fold stratified CV, accuracy)

In [2]:
space = {"n_estimators": [200, 400, 600], "max_depth": [None, 8, 12, 16, 24], "min_samples_leaf": [1, 2, 3, 5, 8],
         "max_features": ["sqrt", "log2", 0.3, 0.5], "criterion": ["gini", "entropy"]}
search = RandomizedSearchCV(RandomForestClassifier(random_state=SEED, n_jobs=2), space, n_iter=20, cv=CV5, scoring="accuracy", random_state=SEED, n_jobs=1).fit(X, y)
r = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
print(r[[c for c in r.columns if c.startswith("param_")] + ["mean_test_score", "std_test_score"]].head(6).round(4).to_string(index=False))

 param_n_estimators  param_min_samples_leaf param_max_features param_max_depth param_criterion  mean_test_score  std_test_score
                400                       1               sqrt              16            gini           0.8848          0.0044
                400                       2                0.5              12         entropy           0.8839          0.0055
                200                       2                0.5              24         entropy           0.8836          0.0062
                400                       5               log2              24         entropy           0.8835          0.0055
                600                       8                0.5               8         entropy           0.8831          0.0053
                400                       2                0.5            None         entropy           0.8827          0.0065


## Stability check, overfitting check, and submission

In [3]:
best = search.best_estimator_
print("best params:", search.best_params_, "| 5-fold CV acc:", round(search.best_score_, 4))
rep = cross_val_score(best, X, y, cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED + 1), scoring="accuracy", n_jobs=1)
oof = cross_val_predict(best, X, y, cv=CV5)
print(f"Repeated CV accuracy: {rep.mean():.4f} ± {rep.std():.4f} | OOF macro-F1: {f1_score(y, oof, average='macro'):.4f}")
best.fit(X, y)
print("train accuracy:", round(accuracy_score(y, best.predict(X)), 4), "vs CV", round(search.best_score_, 4))
pred = best.predict(X_test).astype(int)
sub = pd.DataFrame({ID: test[ID], TARGET: pred})
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample) and (sub[ID].values == sample[ID].values).all() and set(sub[TARGET]) <= {0, 1, 2}
sub.to_csv("submissions/submission_random_forest.csv", index=False)
json.dump(dict(model="Random Forest", cv_acc=float(search.best_score_), repeated_cv=float(rep.mean()), repeated_cv_std=float(rep.std()),
               macro_f1=float(f1_score(y, oof, average="macro")), train_acc=float(accuracy_score(y, best.predict(X))), params=str(search.best_params_)), open("submissions/submission_random_forest_result.json", "w"))
print("written submission_random_forest.csv | predicted class counts:", np.bincount(pred, minlength=3).tolist())

best params: {'n_estimators': 400, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 16, 'criterion': 'gini'} | 5-fold CV acc: 0.8848


Repeated CV accuracy: 0.8840 ± 0.0052 | OOF macro-F1: 0.8854


train accuracy: 0.9963 vs CV 0.8848


written submission_random_forest.csv | predicted class counts: [839, 831, 830]
